# Tema 3.1 – Estadística descriptiva con Pandas (World Happiness 2015–2024)

En este notebook se trabajará la **estadística descriptiva básica** aplicada al dataset de *World Happiness*, unificándolo para el periodo **2015–2024**.

Usaremos como eje principal la variable de felicidad (`Happiness score`) y varios factores asociados:

- **Happiness score** (felicidad percibida)
- **GDP per capita** (PIB per cápita)
- **Social support** (apoyo social percibido)
- **Healthy life expectancy** (esperanza de vida saludable)
- **Freedom to make life choices** (libertad para tomar decisiones)
- **Generosity** (generosidad)
- **Perceptions of corruption** (percepción de corrupción)

Los datos combinados contienen:
- **1502 registros** correspondientes a observaciones de país-año.
- **175 países distintos** a lo largo del periodo.

## 🎯 Pregunta guía principal

> **¿Cómo se comporta la felicidad global y sus factores entre 2015 y 2024?**

### ✅ Respuesta (a alto nivel)

A nivel global, la felicidad promedio en el mundo entre **2015 y 2023** se sitúa alrededor de **5.44 puntos** (en una escala de 0 a 10), con una mediana de **5.43** y una desviación estándar de aproximadamente **1.12**. 
Esto indica un nivel de felicidad **moderado**, con diferencias importantes entre países (el valor mínimo observado es **1.86** y el máximo **7.84**).
En la sesión, iremos profundizando en cómo se distribuye esta felicidad por año y por región.


## 0. Carga de librerías y unificación de los datos (2015–2024)

### 🧠 ¿Por qué unificar los años?

En lugar de analizar un solo año aislado, al unir todos los archivos de **2015 a 2024** podemos responder preguntas mucho más ricas, por ejemplo:

- ¿Ha cambiado la felicidad promedio del mundo en estos 10 años?
- ¿La desigualdad entre países (en términos de felicidad) aumenta o disminuye?
- ¿Los factores asociados (PIB, apoyo social, salud, libertad, etc.) se mantienen estables o cambian con el tiempo?

Para esto, primero unificamos todos los años en un solo `DataFrame` largo, agregando una columna `Year` para saber a qué año pertenece cada registro.


In [1]:
print('Hola mundo!')

Hola mundo!


Instalar librerías necesarias

In [51]:
#!pip install matplotlib seaborn plotly numpy pandas missingno

In [2]:
# =========================
# 0.1 Importar librerías
# =========================

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)


In [6]:
# =========================
# 0.2 Cargar y unificar los datasets 2015–2024
# =========================

years = range(2015, 2025)  # 2015 a 2024
dfs = []

for year in years:
    file_name = f"./data/world_happiness_{year}.csv"  # Ajusta el nombre si es necesario

    df_year = pd.read_csv(
        file_name,
        sep=";",              # Los archivos usan ';' como separador
        engine="python",      # Motor más flexible
        on_bad_lines="skip"    # Ignora filas defectuosas para evitar errores
    )

    df_year["Year"] = year
    dfs.append(df_year)


df = pd.concat(dfs, ignore_index=True)

df.head(10)

,Ranking,Country,Regional indicator,Happiness score,GDP per capita,Social support,Healthy life expectancy,Freedom to make life choices,Generosity,Perceptions of corruption,Year
0,1,Switzerland,Western Europe,7.59,8.26,0.96,73.0,0.99,0.37,0.24,2015
1,2,Iceland,Western Europe,7.56,7.70,1.00,73.0,0.94,0.55,0.74,2015
2,3,Denmark,Western Europe,7.53,7.84,0.97,70.0,0.97,0.43,0.12,2015
3,4,Norway,Western Europe,7.52,8.63,0.95,71.0,1.00,0.44,0.34,2015
4,5,Canada,North America and ANZ,7.43,7.85,0.94,71.0,0.95,0.58,0.40,2015
5,6,Finland,Western Europe,7.41,7.63,0.94,71.0,0.96,0.29,0.25,2015
6,7,Netherlands,Western Europe,7.38,7.86,0.91,71.0,0.92,0.60,0.42,2015
7,8,Sweden,Western Europe,7.36,7.88,0.92,72.0,0.99,0.46,0.21,2015
8,9,New Zealand,North America and ANZ,7.29,7.40,0.94,72.0,0.95,0.60,0.22,2015
9,10,Australia,North America and ANZ,7.28,7.89,0.93,72.0,0.97,0.55,0.35,2015


### 0.3 Limpieza inicial: columnas y formato numérico de la felicidad

Vamos a trabajar con **`Happiness score`**, pero al leer el archivo viene con **coma como separador decimal** (ejemplo: `7,59`).

Por eso, primero convertimos ese texto a número de punto flotante.


In [7]:
# =========================
# 0.3 Crear una columna numérica limpia de felicidad
# =========================

# Creamos una nueva columna numérica a partir de 'Happiness score'
df["Happiness_score_num"] = (
    df["Happiness score"]
    .astype(str)
    .str.replace(",", ".", regex=False)
)

df["Happiness_score_num"] = pd.to_numeric(df["Happiness_score_num"], errors="coerce")

# Renombramos algunas columnas para trabajar más cómodo
df = df.rename(columns={
    "Regional indicator": "Region",
    "GDP per capita": "GDP_per_capita",
    "Social support": "Social_support",
    "Healthy life expectancy": "Healthy_life_expectancy",
    "Freedom to make life choices": "Freedom",
    "Perceptions of corruption": "Corruption",
})

df.head()

,Ranking,Country,Region,Happiness score,GDP_per_capita,Social_support,Healthy_life_expectancy,Freedom,Generosity,Corruption,Year,Happiness_score_num
0,1,Switzerland,Western Europe,7.59,8.26,0.96,73.0,0.99,0.37,0.24,2015,7.59
1,2,Iceland,Western Europe,7.56,7.70,1.00,73.0,0.94,0.55,0.74,2015,7.56
2,3,Denmark,Western Europe,7.53,7.84,0.97,70.0,0.97,0.43,0.12,2015,7.53
3,4,Norway,Western Europe,7.52,8.63,0.95,71.0,1.00,0.44,0.34,2015,7.52
4,5,Canada,North America and ANZ,7.43,7.85,0.94,71.0,0.95,0.58,0.40,2015,7.43


## 1. ¿Qué es la estadística descriptiva?

La **estadística descriptiva** permite **resumir y describir** un conjunto de datos mediante números clave:

- **Medidas de tendencia central**: valores “típicos” (media, mediana, moda).
- **Medidas de dispersión**: qué tan esparcidos están los datos (rango, varianza, desviación estándar).

### ¿Para qué la usamos en este contexto?

En el contexto del *World Happiness Report* la estadística descriptiva nos ayuda a responder preguntas como:

- **¿Cuál ha sido el nivel típico de felicidad en el mundo entre 2015 y 2024?**
- **¿Hay mucha desigualdad en felicidad entre países?**
- **¿Los factores como el PIB, el apoyo social o la salud muestran mucha variación entre países?**

Antes de hacer modelos más complejos o visualizaciones avanzadas, es fundamental tener esta “fotografía numérica” del dataset.

### Respuesta (resumen global)

En la muestra combinada 2015–2024, la felicidad promedio global es de **5.44**, con una mediana de **5.43**. 
La desviación estándar de **1.12** indica una **variación considerable** entre países, y los valores observados van desde **1.86** hasta **7.84** puntos.


### 1.1 Resumen rápido con `describe()`

La función `describe()` en Pandas permite obtener de forma rápida varias medidas descriptivas para las columnas numéricas:

- `count`: cantidad de datos no nulos  
- `mean`: media  
- `std`: desviación estándar  
- `min`, `25%`, `50%`, `75%`, `max`: mínimo, cuartiles y máximo  

### ¿Cómo ayuda `describe()` a nuestras preguntas?

Con un solo comando, podemos ver:

- En qué rango se mueve la felicidad (`Happiness_score`).
- Cuál es el promedio y la dispersión de PIB per cápita, apoyo social, salud, etc.
- Si hay valores muy extremos (por ejemplo, países con PIB muy alto frente a otros muy bajo).


In [8]:
# =========================
# 1.1 Resumen descriptivo global de variables clave
# =========================

cols_interest = [
    "Happiness_score_num",
    "GDP_per_capita",
    "Social_support",
    "Healthy_life_expectancy",
    "Freedom",
    "Generosity",
    "Corruption",
]
#df.describe()
df[cols_interest].describe()

,Happiness_score_num,GDP_per_capita,Social_support,Healthy_life_expectancy,Freedom,Generosity,Corruption
count,1502.000,1502.000,1502.000,1501.000,1502.000,1502.000,1502.000
mean,5.449,6.107,0.692,66.689,0.659,0.320,0.453
std,1.126,2.500,0.213,7.641,0.216,0.173,0.322
min,1.721,0.000,0.000,39.000,0.000,0.000,0.000
25%,4.593,4.376,0.565,62.000,0.536,0.196,0.159
50%,5.469,6.306,0.739,68.000,0.690,0.296,0.345
75%,6.278,8.050,0.861,72.000,0.831,0.430,0.783
max,7.842,10.000,1.000,85.000,1.000,1.000,1.000


📝 **Guía para interpretar en clase**

- Observa el promedio (`mean`) de `Happiness_score_num`: te da una idea de la **felicidad típica** en el mundo.
- La desviación estándar (`std`) indica qué tan diferentes son los países entre sí.
- Revisa los valores mínimos y máximos para ver **extremos** (países muy felices vs. muy infelices).
- Compara la dispersión de `GDP_per_capita`, `Social_support` y `Healthy_life_expectancy` para ver qué factor varía más entre países.


### 1.2 Medidas de tendencia central para la felicidad

Ahora profundizamos en la variable principal: `Happiness_score_num` (felicidad).

### ¿Por qué calcular media y mediana?

- La **media** nos dice el nivel promedio de felicidad.
- La **mediana** indica el valor central: la mitad de los países están por encima y la otra mitad por debajo.

Al comparar media y mediana, podemos intuir si la distribución es **simétrica** o está sesgada por países muy extremos.

### Respuesta

La media (**5.44**) y la mediana (**5.43**) son muy similares, lo que sugiere que **no hay un sesgo extremo** hacia valores muy altos o muy bajos de felicidad a nivel global. La distribución, en conjunto, es relativamente equilibrada.


In [9]:
# =========================
# 1.2 Medidas de tendencia central de la felicidad
# =========================

col = "Happiness_score_num"

mean_hap = df[col].mean()
median_hap = df[col].median()
mode_hap = df[col].mode()

mean_hap, median_hap, mode_hap

(np.float64(5.448899800266311),
 np.float64(5.46865),
 0    3.775
 1    4.510
 2    5.190
 3    5.890
 4    6.192
 5    6.300
 Name: Happiness_score_num, dtype: float64)

### 1.3 Medidas de dispersión: ¿qué tan desiguales son los países?

Las medidas de dispersión nos indican qué tan homogéneos o desiguales son los niveles de felicidad entre los países.

- **Rango**: diferencia entre el país más feliz y el menos feliz.
- **Varianza** y **desviación estándar**: qué tanto se alejan, en promedio, los países de la media.

### ¿Cómo ayuda esto a entender el mundo?

- Un **rango** muy alto implica que existen países muy felices y otros muy infelices.
- Una **desviación estándar** alta indica que la felicidad está muy dispersa entre países.

### Respuesta

El rango de felicidad va de **1.86** a **7.84** puntos, con una desviación estándar de aproximadamente **1.12**. 
Esto confirma que hay **diferencias marcadas** entre países: algunos se encuentran en la parte alta del ranking global de bienestar, mientras que otros están claramente rezagados.


In [10]:
# =========================
# 1.3 Medidas de dispersión de la felicidad
# =========================

value_min = df[col].min()
value_max = df[col].max()
value_range = value_max - value_min
variance = df[col].var()
std_dev = df[col].std()

#value_min, value_max, value_range, variance, std_dev
print(f"""
Estadísticas descriptivas para la columna: **{col}**

• Valor mínimo:          {value_min:.3f}
• Valor máximo:          {value_max:.3f}
• Rango (max - min):     {value_range:.3f}
• Varianza:              {variance:.3f}
• Desviación estándar:   {std_dev:.3f}

""")


Estadísticas descriptivas para la columna: **Happiness_score_num**

• Valor mínimo:          1.721
• Valor máximo:          7.842
• Rango (max - min):     6.121
• Varianza:              1.267
• Desviación estándar:   1.126




## 2. Ejemplo guiado: felicidad promedio y dispersión por año

Hasta ahora hemos visto estadísticas globales (todos los años juntos). Ahora queremos acercarnos más a la pregunta:

> **¿La felicidad global ha cambiado entre 2015 y 2024?**

Para esto, calcularemos estadísticas descriptivas **por año**.


In [11]:
# =========================
# 2.1 Estadística descriptiva por año
# =========================

year_summary = df.groupby("Year")[col].agg(["count", "mean", "median", "std", "min", "max"])
year_summary

,count,mean,median,std,min,max
Year,,,,,,
2015,158,5.376,5.230,1.145,2.840,7.590
2016,157,5.382,5.314,1.142,2.905,7.526
2017,155,5.354,5.279,1.131,2.693,7.537
2018,155,5.373,5.358,1.123,2.905,7.632
2019,155,5.405,5.373,1.116,2.853,7.769
2020,152,5.473,5.513,1.116,2.567,7.809
2021,148,5.533,5.505,1.078,2.523,7.842
2022,145,5.554,5.578,1.091,2.404,7.821
2023,137,5.540,5.684,1.140,1.859,7.804


### Interpretación y respuesta

- En **2015**, la felicidad promedio fue de aproximadamente **5.38** puntos.
- En **2023**, la felicidad promedio se sitúa cerca de **5.54** puntos.
- El año con la media más alta en el periodo es **2022**, con un promedio de **5.55**.

En general, se observa una **ligera tendencia al alza** en la felicidad promedio global entre 2015 y 2023, sin cambios drásticos, pero sí con un leve incremento hacia los años más recientes.

> Esto sugiere que, pese a crisis puntuales (económicas, sanitarias, etc.), el nivel promedio de felicidad reportada se ha mantenido **estable con una tendencia suavemente positiva**.


## 3. Felicidad por región

Otra forma muy útil de aplicar estadística descriptiva es agrupar por región y analizar los promedios y la dispersión por zona geográfica.

### Pregunta clave

> **¿Qué regiones del mundo son, en promedio, más felices y cuáles están más rezagadas?**

### Respuesta basada en los datos

- La región con mayor felicidad promedio es **North America and ANZ**, con una media de **7.18** puntos.
- Le sigue **Western Europe**, con un promedio de **6.71** puntos.
- En el otro extremo, la región con menor felicidad promedio es **Sub-Saharan Africa**, con **4.31** puntos.

Esto refleja **brechas claras** en bienestar subjetivo entre regiones: las economías más desarrolladas tienden a reportar niveles más altos de felicidad, mientras que regiones con mayores desafíos estructurales obtienen puntajes más bajos.


In [ ]:
# =========================
# 3.1 Estadística descriptiva de felicidad por región
# =========================

region_summary = df.groupby("Region")[col].agg(["count", "mean", "median", "std"]).sort_values("mean", ascending=False)
region_summary

,count,mean,median,std
Region,,,,
North America and ANZ,40,7.151,7.192,0.172
Western Europe,232,6.723,6.870,0.744
Latin America and Caribbean,209,6.008,6.105,0.635
Central and Eastern Europe,141,5.743,5.766,0.582
East Asia,68,5.691,5.789,0.498
Southeast Asia,82,5.407,5.384,0.734
Commonwealth of Independent States,103,5.366,5.455,0.550
Middle East and North Africa,189,5.246,5.129,1.054
South Asia,65,4.398,4.500,0.853


## 4. Tu turno – Práctica guiada 

Ahora es tu momento de aplicar lo aprendido. Completa los siguientes ejercicios utilizando el DataFrame `df`.

> Recuerda: puedes adaptar los nombres de columnas si tu versión del dataset los tiene ligeramente diferentes.


### Ejercicio 1 – Comparar factores clave

Elige **dos variables numéricas** (por ejemplo, `GDP_per_capita` y `Social_support`) y:

1. Usa `describe()` para cada una por separado.
2. Compara sus medias y desviaciones estándar.
3. Escribe una breve interpretación sobre cuál muestra más desigualdad entre países.


In [15]:
# ==== Ejercicio 1 – Tu solución aquí ====
GDP_SocialSupport_compare = [
    "GDP_per_capita",
    "Social_support"
]
df[GDP_SocialSupport_compare].describe()

,GDP_per_capita,Social_support
count,1502.000,1502.000
mean,6.107,0.692
std,2.500,0.213
min,0.000,0.000
25%,4.376,0.565
50%,6.306,0.739
75%,8.050,0.861
max,10.000,1.000


Interpretación:

Social Support muestra más desigualdad entre países

### Ejercicio 2 – Felicidad por región

1. Calcula media, mediana y desviación estándar de `Happiness_score` **por región**.
2. Identifica la región:
   - Más feliz en promedio.
   - Con mayor dispersión de felicidad entre países.
3. Escribe una reflexión corta sobre estas diferencias.


In [19]:
# ==== Ejercicio 2 – Tu solución aquí ====
region_summary2 = df.groupby("Region")[col].agg(["count", "mean", "median", "std"]).sort_values("mean", ascending=False)
region_summary2

,count,mean,median,std
Region,,,,
North America and ANZ,40,7.151,7.192,0.172
Western Europe,232,6.723,6.870,0.744
Latin America and Caribbean,209,6.008,6.105,0.635
Central and Eastern Europe,141,5.743,5.766,0.582
East Asia,68,5.691,5.789,0.498
Southeast Asia,82,5.407,5.384,0.734
Commonwealth of Independent States,103,5.366,5.455,0.550
Middle East and North Africa,189,5.246,5.129,1.054
South Asia,65,4.398,4.500,0.853


2. *  La región con el mejor promedio de felicidad es North America and ANZ
    *  La región con mayor dispersión de felicidad entre paises es Middle East and North Africa

3. El oeste de europa y la zona sur (sur y sureste) de Asia es donde se presenta más diferencia en la dispersión de felicidad, sin embargo sus promedios de felicidad no se relacionan mucho,

**Pistas para la interpretación (docente):**

- Esperamos ver a **North America and ANZ** y **Western Europe** en la parte alta del ranking de felicidad.
- Regiones como **Sub-Saharan Africa** tenderán a estar en la parte baja.
- La desviación estándar por región ayuda a ver si dentro de una misma región hay países muy dispares o si los niveles de felicidad son relativamente homogéneos.


### Ejercicio 3 – Evolución por año de un factor

Elige una variable diferente a la felicidad, por ejemplo `Healthy_life_expectancy` o `Freedom`, y:

1. Calcula la media por año.
2. Observa si esa media sube, baja o se mantiene relativamente estable.
3. Relaciona brevemente tu hallazgo con lo que sabes del contexto mundial en esos años (crisis, pandemia, etc.).


In [21]:
# ==== Ejercicio 3 – Tu solución aquí ====

factor = "Freedom"
freedom_by_year = df.groupby('Year')[factor].mean()

freedom_by_year

Year
2015    0.640
2016    0.610
2017    0.621
2018    0.630
2019    0.622
2020    0.669
2021    0.696
2022    0.699
2023    0.700
2024    0.719
Name: Freedom, dtype: float64

2. Sube un poco pero se mantiene relativamente estable

**Conclusiones de la sesión:**

Escribe al menos **3 conclusiones** que puedas extraer del análisis descriptivo hecho hoy. Por ejemplo:

- Comportamiento general de la felicidad en el mundo.
- Factores que muestran más desigualdad entre países.
- Alguna intuición inicial sobre cómo han cambiado las cosas entre 2015 y 2024.
